In [1]:
!pip install anndata==0.8.0

In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
import scipy

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('../../parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [59]:
fn = '../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad'

In [60]:
sam = SAM()
sam.load_data(fn)

In [62]:
lsaturn_mv = pd.read_csv('../../SATURN_mapping/DR_vole_mouse_SATURN_30_seed_07202026.csv', index_col = 'Unnamed: 0')
lsaturn_m = pd.read_csv('../../SATURN_mapping/mouse_zebrafish_30_seeds.csv', index_col = 'barcode')

In [64]:
ind = pd.read_csv('../../Active_SAMap_Joined/DR_MG_mapping_cleaned_07172026_0.csv')['Unnamed: 0']
lsamap_m = pd.DataFrame(index = list(ind))
for i in range(30):
    df = pd.read_csv('../../Active_SAMap_Joined/DR_MG_mapping_cleaned_07172026_'+str(i)+'.csv', index_col = 'Unnamed: 0')
    lsamap_m = pd.concat([lsamap_m, df], axis = 1)

In [65]:
lsamap_m

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
AAACCTGAGAGTTGGC,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,...,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,319 Astro-TE NN
AAACCTGAGGTGACCA,128 VMH Fezf1 Glut,129 VMH Nr5a1 Glut,129 VMH Nr5a1 Glut,113 MEA-COA-BMA Ccdc42 Glut,113 MEA-COA-BMA Ccdc42 Glut,128 VMH Fezf1 Glut,129 VMH Nr5a1 Glut,113 MEA-COA-BMA Ccdc42 Glut,128 VMH Fezf1 Glut,129 VMH Nr5a1 Glut,...,113 MEA-COA-BMA Ccdc42 Glut,129 VMH Nr5a1 Glut,129 VMH Nr5a1 Glut,121 MEA-BST Otp Zic2 Glut,113 MEA-COA-BMA Ccdc42 Glut,129 VMH Nr5a1 Glut,129 VMH Nr5a1 Glut,113 MEA-COA-BMA Ccdc42 Glut,129 VMH Nr5a1 Glut,128 VMH Fezf1 Glut
AAACCTGCAAGCGTAG,328 OEC NN,320 Astro-OLF NN,038 DG-PIR Ex IMN,320 Astro-OLF NN,045 OB-STR-CTX Inh IMN,320 Astro-OLF NN,328 OEC NN,328 OEC NN,328 OEC NN,045 OB-STR-CTX Inh IMN,...,045 OB-STR-CTX Inh IMN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,038 DG-PIR Ex IMN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN,320 Astro-OLF NN
AAACCTGCAGTAGAGC,128 VMH Fezf1 Glut,129 VMH Nr5a1 Glut,129 VMH Nr5a1 Glut,113 MEA-COA-BMA Ccdc42 Glut,113 MEA-COA-BMA Ccdc42 Glut,128 VMH Fezf1 Glut,129 VMH Nr5a1 Glut,113 MEA-COA-BMA Ccdc42 Glut,128 VMH Fezf1 Glut,129 VMH Nr5a1 Glut,...,113 MEA-COA-BMA Ccdc42 Glut,129 VMH Nr5a1 Glut,129 VMH Nr5a1 Glut,121 MEA-BST Otp Zic2 Glut,113 MEA-COA-BMA Ccdc42 Glut,129 VMH Nr5a1 Glut,129 VMH Nr5a1 Glut,113 MEA-COA-BMA Ccdc42 Glut,129 VMH Nr5a1 Glut,128 VMH Fezf1 Glut
AAACCTGCATCACCCT,038 DG-PIR Ex IMN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,038 DG-PIR Ex IMN,045 OB-STR-CTX Inh IMN,038 DG-PIR Ex IMN,038 DG-PIR Ex IMN,038 DG-PIR Ex IMN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,...,038 DG-PIR Ex IMN,038 DG-PIR Ex IMN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,038 DG-PIR Ex IMN,045 OB-STR-CTX Inh IMN,038 DG-PIR Ex IMN,038 DG-PIR Ex IMN,038 DG-PIR Ex IMN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCACACACCGCA,160 PAG-SC Neurod2 Meis2 Glut,161 PAG Pou4f3 Glut,160 PAG-SC Neurod2 Meis2 Glut,160 PAG-SC Neurod2 Meis2 Glut,178 SCig Foxb1 Otx2 Glut,160 PAG-SC Neurod2 Meis2 Glut,178 SCig Foxb1 Otx2 Glut,178 SCig Foxb1 Otx2 Glut,160 PAG-SC Neurod2 Meis2 Glut,160 PAG-SC Neurod2 Meis2 Glut,...,160 PAG-SC Neurod2 Meis2 Glut,160 PAG-SC Neurod2 Meis2 Glut,160 PAG-SC Neurod2 Meis2 Glut,221 LDT-PCG Vsx2 Lhx4 Glut,161 PAG Pou4f3 Glut,230 PRNr Otp Nfib Glut,160 PAG-SC Neurod2 Meis2 Glut,160 PAG-SC Neurod2 Meis2 Glut,178 SCig Foxb1 Otx2 Glut,160 PAG-SC Neurod2 Meis2 Glut
TTTGTCACATGTAAGA,045 OB-STR-CTX Inh IMN,320 Astro-OLF NN,038 DG-PIR Ex IMN,320 Astro-OLF NN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,038 DG-PIR Ex IMN,320 Astro-OLF NN,038 DG-PIR Ex IMN,...,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,320 Astro-OLF NN,045 OB-STR-CTX Inh IMN,038 DG-PIR Ex IMN,038 DG-PIR Ex IMN,320 Astro-OLF NN,045 OB-STR-CTX Inh IMN,045 OB-STR-CTX Inh IMN,320 Astro-OLF NN
TTTGTCACATTATCTC,085 SI-MPO-LPO Lhx8 Gaba,085 SI-MPO-LPO Lhx8 Gaba,085 SI-MPO-LPO Lhx8 Gaba,085 SI-MPO-LPO Lhx8 Gaba,085 SI-MPO-LPO Lhx8 Gaba,085 SI-MPO-LPO Lhx8 Gaba,086 MPO-ADP Lhx8 Gaba,058 PAL-STR Gaba-Chol,085 SI-MPO-LPO Lhx8 Gaba,058 PAL-STR Gaba-Chol,...,086 MPO-ADP Lhx8 Gaba,058 PAL-STR Gaba-Chol,085 SI-MPO-LPO Lhx8 Gaba,058 PAL-STR Gaba-Chol,085 SI-MPO-LPO Lhx8 Gaba,058 PAL-STR Gaba-Chol,055 STR Lhx8 Gaba,085 SI-MPO-LPO Lhx8 Gaba,058 PAL-STR Gaba-Chol,058 PAL-STR Gaba-Chol
TTTGTCAGTCCGAGTC-2,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,...,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,107 DMH Hmx2 Gaba,077 CEA-BST Gal Avp Gaba,10

In [67]:
meta_folder = '../../Active_SAMap_Joined/DR_metadata/'

In [68]:
mn = os.listdir('../../Active_SAMap_Joined/DR_metadata/')

In [69]:
pd.read_csv(meta_folder + mn[3])

,Unnamed: 0,n_genes,n_counts,key,leiden_clusters,subclass_id_label_mapping,subclass_id_label_lc
0,AAACCTGAGAGTTGGC,496,866.0,DarioRerio1,83,Unlabeled,83
1,AAACCTGAGGTGACCA,491,618.0,DarioRerio1,2,Unlabeled,2
2,AAACCTGCAAGCGTAG,774,1502.0,DarioRerio1,200,Unlabeled,200
3,AAACCTGCAGTAGAGC,929,1417.0,DarioRerio1,2,Unlabeled,2
4,AAACCTGCATCACCCT,702,1262.0,DarioRerio1,78,Unlabeled,78
...,...,...,...,...,...,...,...
61425,TTTGTCACACACCGCA,658,1277.0,DanioRerio16,440,Unlabeled,440
61426,TTTGTCACATGTAAGA,2099,8522.0,DanioRerio16,260,Unlabeled,260
61427,TTTGTCACATTATCTC,945,1461.0,DanioRerio16,21,Unlabeled,21
61428,TTTGTCAGTCCGAGTC-2,1271,2548.0,DanioRerio16,134,Unlabeled,134


In [70]:
test = pd.read_csv(meta_folder + mn[0])['subclass_id_label_mapping']

In [71]:
barcodes = pd.read_csv(meta_folder + mn[0])['Unnamed: 0']

In [72]:
raw_lc = pd.read_csv(meta_folder + mn[0])['subclass_id_label_lc']

In [73]:
df_lc = pd.DataFrame(data = list(raw_lc), index = list(barcodes), columns = ['raw_lc'])

In [74]:
mode_df = pd.DataFrame(index = [a for a in range(len(test))], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    print(mn[j])
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_lc'])
    mode_df.loc[:,j] = dat

DR_metadata_subclass_250_cleaned_07202026_24.csv
DR_metadata_subclass_250_cleaned_07202026_12.csv
DR_metadata_subclass_250_cleaned_07202026_5.csv
DR_metadata_subclass_250_cleaned_07202026_4.csv
DR_metadata_subclass_250_cleaned_07202026_1.csv
DR_metadata_subclass_250_cleaned_07202026_17.csv
DR_metadata_subclass_250_cleaned_07202026_15.csv
DR_metadata_subclass_250_cleaned_07202026_0.csv
DR_metadata_subclass_250_cleaned_07202026_20.csv
DR_metadata_subclass_250_cleaned_07202026_11.csv
DR_metadata_subclass_250_cleaned_07202026_6.csv
DR_metadata_subclass_250_cleaned_07202026_26.csv
DR_metadata_subclass_250_cleaned_07202026_9.csv
DR_metadata_subclass_250_cleaned_07202026_2.csv
DR_metadata_subclass_250_cleaned_07202026_7.csv
DR_metadata_subclass_250_cleaned_07202026_29.csv
DR_metadata_subclass_250_cleaned_07202026_28.csv
DR_metadata_subclass_250_cleaned_07202026_16.csv
DR_metadata_subclass_250_cleaned_07202026_25.csv
DR_metadata_subclass_250_cleaned_07202026_21.csv
DR_metadata_subclass_250_cle

In [75]:
fin = []
for item in mode_df.columns:
    fin.append(mode_df.loc[10000,item])

In [76]:
scipy.stats.mode(fin)

ModeResult(mode=array([354]), count=array([30]))

In [77]:
import re

mn = sorted(mn, key=lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)])

In [78]:
mn

['DR_metadata_subclass_250_cleaned_07202026_0.csv',
 'DR_metadata_subclass_250_cleaned_07202026_1.csv',
 'DR_metadata_subclass_250_cleaned_07202026_2.csv',
 'DR_metadata_subclass_250_cleaned_07202026_3.csv',
 'DR_metadata_subclass_250_cleaned_07202026_4.csv',
 'DR_metadata_subclass_250_cleaned_07202026_5.csv',
 'DR_metadata_subclass_250_cleaned_07202026_6.csv',
 'DR_metadata_subclass_250_cleaned_07202026_7.csv',
 'DR_metadata_subclass_250_cleaned_07202026_8.csv',
 'DR_metadata_subclass_250_cleaned_07202026_9.csv',
 'DR_metadata_subclass_250_cleaned_07202026_10.csv',
 'DR_metadata_subclass_250_cleaned_07202026_11.csv',
 'DR_metadata_subclass_250_cleaned_07202026_12.csv',
 'DR_metadata_subclass_250_cleaned_07202026_13.csv',
 'DR_metadata_subclass_250_cleaned_07202026_14.csv',
 'DR_metadata_subclass_250_cleaned_07202026_15.csv',
 'DR_metadata_subclass_250_cleaned_07202026_16.csv',
 'DR_metadata_subclass_250_cleaned_07202026_17.csv',
 'DR_metadata_subclass_250_cleaned_07202026_18.csv',
 'D

In [79]:
lsamap_mv = pd.DataFrame(index = [a for a in barcodes], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_mapping'])
    lsamap_mv.loc[:,j] = dat

In [57]:
#for quail
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '067 LSX Sall3 Pax6 Gaba':
            inp = ['cj_m066_m067']
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['cj_m136_m138']
        if inp[0] == '099 SBPV-PVa Six6 Satb2 Gaba':
            inp = ['cj_m091_m099']
        b += len(barcodes)
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [27]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [28]:
b/len(sam.adata)

0.013791166162818695

In [29]:
a/len(sam.adata)

0.644546403060542

In [30]:
len(samap_barcodes)/len(sam.adata)

0.1832312260895155

In [31]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.12872200968458225

In [40]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.4126323914068865

In [32]:
len(saturn_barcodes)/len(sam.adata)

0.0066882474116482515

In [33]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [34]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

076 MEA-BST Lhx6 Nfib Gaba
135 STN-PSTN Pitx2 Glut
107 DMH Hmx2 Gaba
097 PVHd-SBPV Six3 Prox1 Gaba
134 PH-ant-LHA Otp Bsx Glut
104 TU-ARH Otp Six6 Gaba


In [35]:
a

6

In [20]:
#for anole
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '086 MPO-ADP Lhx8 Gaba':
            inp = ['ac_m058_m086']
        if inp[0] == '124 MPN-MPO-PVpo Hmx2 Glut':
            inp = ['ac_m124_m130']
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['ac_m136_m138']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [83]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [84]:
b/len(sam.adata)

0.02035490605427975

In [85]:
a/len(sam.adata)

0.6907306889352819

In [86]:
len(samap_barcodes)/len(sam.adata)

0.13872651356993737

In [87]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.1363465553235908

In [88]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.4956739526411657

In [89]:
len(saturn_barcodes)/len(sam.adata)

0.013841336116910229

In [222]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [223]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

098 AHN-SBPV-PVHd Pdrm12 Gaba
099 SBPV-PVa Six6 Satb2 Gaba
122 LHA-MEA Otp Glut
117 LHA Barhl2 Glut
118 ADP-MPO Trp73 Glut
073 MEA-BST Sox6 Gaba


In [224]:
a

6

In [48]:
#for xenopus
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '104 TU-ARH Otp Six6 Gaba':
            inp = ['xt_m098_m104']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [179]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [180]:
b/len(sam.adata)

0.0018947916913384334

In [181]:
a/len(sam.adata)

0.582151062267592

In [182]:
len(samap_barcodes)/len(sam.adata)

0.2266644560763601

In [183]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.1718339215082542

In [184]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.4312035661218425

In [185]:
len(saturn_barcodes)/len(sam.adata)

0.011795078278581748

In [186]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [187]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

088 BST Tac2 Gaba
107 DMH Hmx2 Gaba
074 MEA-BST Lhx6 Sp9 Gaba
090 BST-MPN Six3 Nrgn Gaba
136 PMv-TMv Pitx2 Glut
134 PH-ant-LHA Otp Bsx Glut
105 TMd-DMH Foxd2 Gaba
073 MEA-BST Sox6 Gaba
143 MM-ant Foxb1 Glut
091 ARH-PVi Six6 Dopa-Gaba


In [188]:
a

10

In [80]:
#for zebrafish
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[1] != inp[2] and inp[1] != 'Unlabeled' and inp[2] != 'Unlabeled':
        print(inp[0],inp[1],inp[2],lc)
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [101]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [102]:
b/len(sam.adata)

0.0

In [103]:
a/len(sam.adata)

0.9239622334364317

In [104]:
len(samap_barcodes)/len(sam.adata)

0.03716425199413967

In [105]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.03887351456942862

In [106]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.5112395632626847

In [107]:
len(saturn_barcodes)/len(sam.adata)

0.0

In [108]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [109]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

086 MPO-ADP Lhx8 Gaba
105 TMd-DMH Foxd2 Gaba
091 ARH-PVi Six6 Dopa-Gaba
092 TMv-PMv Tbx3 Hist-Gaba


In [110]:
a

4

In [81]:
new_mapping = []
new_mapping_name = 'test'
for item in sam.adata.obs['eq_subclass_lc']:
    new_mapping.append(mapping_dict[item])
sam.adata.obs[new_mapping_name] = new_mapping

In [82]:
a = 0
fin_obs = []
for item in sam.adata.obs_names:
    if sam.adata.obs.loc[item,'test'] == sam.adata.obs.loc[item,'ss_subclass']:
        a += 1
        fin_obs.append(item)

In [83]:
a

61430

In [84]:
len(sam.adata)

61430

In [58]:
sam.save_anndata(fn)